# Day 40 — Practice 4 & 5: Hive External Table + Koneksi DBeaver

**Tujuan:** Buat Hive External Table di atas data Parquet yang sudah ada di HDFS, lalu query dari DBeaver.

Alur:
```
HDFS Parquet  →  Hive External Table  →  Query via DBeaver / Spark SQL
(Data Lake)       (Data Warehouse)         (BI / Analytics Tool)
```

**Konsep yang dipraktikkan:**
- Perbedaan **Managed Table** vs **External Table** di Hive
- Buat External Table yang pointnya ke data HDFS
- `MSCK REPAIR TABLE` untuk detect partisi baru
- Koneksi DBeaver ke HiveServer2
- Query Hive dari DBeaver

> **Prasyarat:** Jalankan dulu `02_adventureworks_to_hdfs.ipynb` agar data sudah ada di HDFS.

## Setup SparkSession dengan Hive Support

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import subprocess

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('Hive-External-Tables') \
    .config('spark.jars', '/home/jovyan/work/jars/postgresql-42.7.3.jar') \
    .config('hive.metastore.uris', 'thrift://hive-metastore:9083') \
    .config('spark.sql.warehouse.dir', 'hdfs://namenode:9000/user/hive/warehouse') \
    .enableHiveSupport() \
    .getOrCreate()

HDFS_BASE = 'hdfs://namenode:9000/datalake/raw'

print(f'Spark {spark.version} ready dengan Hive support')

# Cek koneksi ke Hive Metastore
spark.sql('SHOW DATABASES').show()

## 1. Konsep: Managed vs External Table

| | Managed Table | External Table |
|---|---|---|
| **Data lokasi** | Di-manage Hive (`/user/hive/warehouse`) | Di lokasi HDFS yang kita tentukan |
| **DROP TABLE** | Data **ikut terhapus** | Data di HDFS **tetap ada** |
| **Use case** | Data yang sepenuhnya dikelola Hive | Data yang dibuat dari proses lain (Spark, Flume, dll) |
| **Kita pakai** | — | ✅ (data sudah di HDFS dari notebook sebelumnya) |

## 2. Buat Database Hive

In [ ]:
# Buat database adventureworks di Hive
spark.sql('CREATE DATABASE IF NOT EXISTS adventureworks COMMENT "AdventureWorks Data Warehouse"')
spark.sql('USE adventureworks')

print('=== Databases yang tersedia ===')
spark.sql('SHOW DATABASES').show()

## 3. Buat External Tables — Tabel Dimensi

Tabel dimensi tidak berpartisi. Langsung point ke folder Parquet.

In [ ]:
# Helper: buat external table dari Parquet di HDFS
def create_external_table(table_name: str, hdfs_path: str):
    """
    Buat Hive External Table yang pointnya ke Parquet di HDFS.
    Schema otomatis di-infer dari file Parquet.
    """
    # Baca schema dari Parquet
    df = spark.read.parquet(hdfs_path)
    
    # Register sebagai temp view dulu
    df.createOrReplaceTempView(f'_tmp_{table_name}')
    
    # Buat external table
    spark.sql(f'DROP TABLE IF EXISTS adventureworks.{table_name}')
    spark.sql(f"""
        CREATE EXTERNAL TABLE adventureworks.{table_name}
        USING PARQUET
        LOCATION '{hdfs_path}'
    """)
    
    count = spark.sql(f'SELECT COUNT(*) FROM adventureworks.{table_name}').collect()[0][0]
    print(f'✓ {table_name:35s} → {count:6,} rows  ({hdfs_path})')

# Buat external table untuk semua tabel dimensi
dim_tables = [
    ('dim_product',           f'{HDFS_BASE}/production/product'),
    ('dim_product_category',  f'{HDFS_BASE}/production/productcategory'),
    ('dim_product_subcat',    f'{HDFS_BASE}/production/productsubcategory'),
    ('dim_person',            f'{HDFS_BASE}/person/person'),
    ('dim_employee',          f'{HDFS_BASE}/humanresources/employee'),
    ('dim_department',        f'{HDFS_BASE}/humanresources/department'),
    ('dim_customer',          f'{HDFS_BASE}/sales/customer'),
    ('dim_territory',         f'{HDFS_BASE}/sales/salesterritory'),
]

for table_name, path in dim_tables:
    create_external_table(table_name, path)

print('\nSemua tabel dimensi berhasil dibuat di Hive!')

## 4. Buat External Tables — Tabel Fakta (dengan Partisi)

In [ ]:
# Tabel fakta salesorderheader — partisi order_year, order_month
orders_path = f'{HDFS_BASE}/sales/salesorderheader'

spark.sql('DROP TABLE IF EXISTS adventureworks.fact_sales_orders')
spark.sql(f"""
    CREATE EXTERNAL TABLE adventureworks.fact_sales_orders
    USING PARQUET
    PARTITIONED BY (order_year INT, order_month INT)
    LOCATION '{orders_path}'
""")

# MSCK REPAIR TABLE: scan HDFS dan auto-register semua partisi ke metastore
print('Mendeteksi partisi di HDFS...')
spark.sql('MSCK REPAIR TABLE adventureworks.fact_sales_orders')

print('\n=== Partisi yang terdeteksi ===')
spark.sql('SHOW PARTITIONS adventureworks.fact_sales_orders').show(30)

count = spark.sql('SELECT COUNT(*) FROM adventureworks.fact_sales_orders').collect()[0][0]
print(f'Total rows: {count:,}')

In [ ]:
# Tabel fakta salesorderdetail — partisi order_year
detail_path = f'{HDFS_BASE}/sales/salesorderdetail'

spark.sql('DROP TABLE IF EXISTS adventureworks.fact_order_details')
spark.sql(f"""
    CREATE EXTERNAL TABLE adventureworks.fact_order_details
    USING PARQUET
    PARTITIONED BY (order_year INT)
    LOCATION '{detail_path}'
""")
spark.sql('MSCK REPAIR TABLE adventureworks.fact_order_details')

count = spark.sql('SELECT COUNT(*) FROM adventureworks.fact_order_details').collect()[0][0]
print(f'✓ fact_order_details: {count:,} rows')

## 5. Verifikasi: Query Hive External Tables

In [ ]:
print('=== Semua tabel di database adventureworks ===')
spark.sql('SHOW TABLES IN adventureworks').show(30)

In [ ]:
# Query: Total revenue per tahun via Hive
spark.sql("""
    SELECT
        order_year,
        COUNT(*)                    AS total_orders,
        ROUND(SUM(subtotal), 2)     AS total_subtotal,
        ROUND(SUM(totaldue), 2)     AS total_revenue
    FROM adventureworks.fact_sales_orders
    GROUP BY order_year
    ORDER BY order_year
""").show()

In [ ]:
# Query join: Top produk terlaris dari Hive
spark.sql("""
    SELECT
        p.name                          AS product_name,
        p.color,
        SUM(od.orderqty)                AS total_qty,
        ROUND(SUM(od.linetotal), 2)     AS total_revenue
    FROM adventureworks.fact_order_details od
    JOIN adventureworks.dim_product p ON od.productid = p.productid
    GROUP BY p.name, p.color
    ORDER BY total_revenue DESC
    LIMIT 10
""").show(truncate=False)

## 6. Koneksi DBeaver ke HiveServer2

Setelah tabel berhasil dibuat di Hive, kamu bisa query langsung dari DBeaver.

### Langkah-langkah:

1. **Buka DBeaver** → New Connection
2. **Pilih** `Apache Hive`
3. **Isi connection settings:**

| Setting | Value |
|---------|-------|
| Host | `localhost` |
| Port | `10000` |
| Database | `adventureworks` |
| Username | *(kosongkan)* |
| Password | *(kosongkan)* |

4. **Test Connection** → jika sukses, klik **Finish**
5. Buka SQL Editor, coba query:

```sql
SHOW TABLES;

SELECT * FROM dim_product LIMIT 10;

SELECT order_year, COUNT(*) AS total_orders
FROM fact_sales_orders
GROUP BY order_year;
```

> **Catatan:** HiveServer2 berjalan di container pada port `10000`. Pastikan container `hiveserver2` sudah berjalan dengan `docker-compose ps`.

In [ ]:
# Verifikasi HiveServer2 berjalan
result = subprocess.run('docker ps | grep hiveserver2', shell=True, capture_output=True, text=True)
if result.stdout:
    print('✅ HiveServer2 berjalan:')
    print(result.stdout)
    print('\nBuka DBeaver dan connect ke:')
    print('  Host: localhost')
    print('  Port: 10000')
    print('  DB  : adventureworks')
else:
    print('❌ HiveServer2 tidak berjalan. Jalankan: docker-compose up -d hiveserver2')

## 7. Bonus: Cek Detail Tabel di Hive

In [ ]:
# Describe detail tabel — lihat bahwa ini EXTERNAL table
spark.sql('DESCRIBE EXTENDED adventureworks.fact_sales_orders').show(40, truncate=False)

In [ ]:
# Buktikan: DROP external table tidak menghapus data di HDFS
# (jalankan ini jika mau uji coba)

# spark.sql('DROP TABLE adventureworks.fact_sales_orders')
#
# Cek HDFS — data masih ada:
# subprocess.run('docker exec hadoop-namenode hdfs dfs -ls /datalake/raw/sales/salesorderheader', shell=True)
#
# Buat ulang tabel:
# spark.sql('CREATE EXTERNAL TABLE ...')

print('Uncomment kode di atas untuk uji coba DROP external table')

## Summary

Yang sudah dipraktikkan:
- ✅ Buat database `adventureworks` di Hive
- ✅ Buat **External Table** (bukan Managed) yang pointnya ke Parquet di HDFS
- ✅ `MSCK REPAIR TABLE` untuk auto-detect partisi dari HDFS
- ✅ Query tabel Hive via Spark SQL
- ✅ Cara koneksi DBeaver ke HiveServer2 (port 10000)

**Arsitektur yang sudah berdiri:**
```
PostgreSQL (Source)  →  HDFS Parquet (Data Lake)  →  Hive (Data Warehouse)
                         /datalake/raw/...             adventureworks.fact_*
                                                        adventureworks.dim_*
```

**Next →** Day 41 — Gunakan Spark untuk transformasi data di Hive.